# Weak-Instrument Frontier | Phase-3 RETRY (NB12 leftovers)

**Notebook:** `NB12R` | **Retries unfinished cells of:** `NB12.ipynb` | repo pinned to `https://github.com/hugogobato/weakiv-frontier.git` @ `main` (sha used by all completed runs)

| cell | experiment | expected wall |
|---|---|---|
| `n2000_a0.5_p2` | phase3_size_grid | ~3.7 h |
| `n2000_a0.5_p25` | phase3_size_grid | ~3.7 h |
| `n2000_a0.5_p5` | phase3_size_grid | ~3.7 h |

Expected wall times come from the already-completed size-grid twins (size-grid compute does not depend on `alpha`, only `n, q, p`). Keep each notebook on its own runtime; total session budget should stay under Colab's limit.

After every finished cell the runner zips `/content/results` and triggers a browser download (`*_ckpt_*` files), so a dropped session only loses the cell in flight.

In [ ]:
import json, os, subprocess, sys, time

REPO_URL = "https://github.com/hugogobato/weakiv-frontier.git"
REPO_REF = "main"
NB_ID = "NB12R"

if not os.path.isdir("weakiv-frontier"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_REF,
                    REPO_URL], check=True)
os.chdir("weakiv-frontier")
GIT_SHA = subprocess.check_output(
    ["git", "rev-parse", "HEAD"]).decode().strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                "./Research/spectraliv"], check=True)
sys.path.insert(0, "Research/spectraliv/src")
print("repo sha:", GIT_SHA)


In [ ]:
import shutil, traceback, time
from spectraliv import __version__ as pkg_version
from spectraliv.experiments import (
    run_size_cell, run_power_cell, run_decisive_cell, run_robust_cell,
    run_scaling,
)

OUT_ROOT = "/content/results"
MASTER_SEED = 20260823

B_PARAMS = {
    "phase3_size_grid": 20000,
    "phase3_power_surface": 300,
    "phase3_decisive_grid": 400,
    "phase3_robustness": 4000,
}
PATCH_PARAMS = {"patch_reps": 250, "b_boot": 99}

CELLS = [
 {
  "experiment": "phase3_size_grid",
  "cell_id": "n2000_a0.5_p2",
  "cell": {
   "n": 2000,
   "p": 2,
   "q": 1000,
   "alpha": 0.5,
   "cell_id": "n2000_a0.5_p2"
  }
 },
 {
  "experiment": "phase3_size_grid",
  "cell_id": "n2000_a0.5_p25",
  "cell": {
   "n": 2000,
   "p": 25,
   "q": 1000,
   "alpha": 0.5,
   "cell_id": "n2000_a0.5_p25"
  }
 },
 {
  "experiment": "phase3_size_grid",
  "cell_id": "n2000_a0.5_p5",
  "cell": {
   "n": 2000,
   "p": 5,
   "q": 1000,
   "alpha": 0.5,
   "cell_id": "n2000_a0.5_p5"
  }
 }
]

def done_markers(exp, cid):
    d = os.path.join(OUT_ROOT, exp, "cells")
    if exp == "phase3_decisive_grid":
        names = ["_done_%s_coverage.csv.json" % cid,
                 "_done_%s_risk.csv.json" % cid]
    else:
        names = ["_done_%s.csv.json" % cid]
    return all(os.path.exists(os.path.join(d, nm)) for nm in names)


def checkpoint_download(tag):
    """Zip everything so far and push it to the browser. A dead session then
    only loses the cell that was in flight, never finished ones."""
    try:
        zpath = "/content/results_%s_ckpt_%s" % (NB_ID, tag)
        shutil.make_archive(zpath, "zip", OUT_ROOT)
        from google.colab import files
        files.download(zpath + ".zip")
        print("[ckpt] downloaded:", zpath + ".zip")
    except Exception as e:
        print("(checkpoint download skipped):", e)


summary = []
for item in CELLS:
    exp, cid = item["experiment"], item["cell_id"]
    if done_markers(exp, cid):
        print("[skip] %s (done markers present)" % cid)
        continue
    t0 = time.time()
    try:
        cell = dict(item["cell"])
        if exp == "phase3_size_grid":
            run_size_cell(cell, big_b=B_PARAMS[exp], b_cal_cv=4000,
                          out_root=OUT_ROOT)
        elif exp == "phase3_power_surface":
            run_power_cell(cell, reps_per_theta=B_PARAMS[exp], b_cal_cv=4000,
                           out_root=OUT_ROOT)
        elif exp == "phase3_decisive_grid":
            run_decisive_cell(cell, reps=B_PARAMS[exp], b_cal_cv=4000,
                              out_root=OUT_ROOT)
        elif exp == "phase3_robustness":
            run_robust_cell(cell, big_b=B_PARAMS[exp],
                            patch_reps=PATCH_PARAMS["patch_reps"],
                            b_boot=PATCH_PARAMS["b_boot"], out_root=OUT_ROOT)
        elif exp == "phase3_scaling":
            run_scaling(out_root=OUT_ROOT)
        print("[done] %s in %.1fs" % (cid, time.time() - t0))
        summary.append([cid, round(time.time() - t0, 1)])
        checkpoint_download(cid.replace(".", ""))
    except Exception:
        print("[FAIL] %s" % cid)
        traceback.print_exc()

manifest = {
    "notebook_id": NB_ID, "git_sha": GIT_SHA,
    "package_version": pkg_version, "master_seed": MASTER_SEED,
    "b_params": B_PARAMS, "patch_params": PATCH_PARAMS,
    "cells": [c["cell_id"] for c in CELLS],
    "timings_s": summary,
}
with open(os.path.join(OUT_ROOT, "manifest_%s.json" % NB_ID), "w") as f:
    json.dump(manifest, f, indent=1)
print(json.dumps(manifest, indent=1))


In [ ]:
import shutil, os
os.chdir("/content")
shutil.make_archive("results_NB12R", "zip", "/content/results")
try:
    from google.colab import files
    files.download("results_NB12R.zip")
    print("Downloaded:", "results_NB12R.zip")
except Exception as e:
    print("(Not on Colab / download skipped):", e)
